<a href="https://colab.research.google.com/github/weagan/Dust/blob/main/backprop_vs_dust_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Side-by-Side Comparison: Backpropagation vs. Dust Optimization

This notebook provides a complete implementation of a simple neural network trained using two different paradigms:
1. **Standard Backpropagation** (First-order optimization)
2. **Dust Alternative** (Zeroth-order optimization via activation-space perturbation)

### Environment Setup & Secrets Access
This notebook reads optional platform secrets securely on both Google Colab and Kaggle.

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# Attempting to load platform-specific secrets securely
HF_TOKEN = None
try:
    # Google Colab Secrets API
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("Successfully accessed HF_TOKEN via Google Colab secrets storage.")
except Exception:
    try:
        # Kaggle Secrets API
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
        print("Successfully accessed HF_TOKEN via Kaggle secrets storage.")
    except Exception:
        print("No environment secret keys detected or accessible. Proceeding using public/local defaults.")

# Set device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using tracking target backend device: {device}")

Successfully accessed HF_TOKEN via Google Colab secrets storage.
Using tracking target backend device: cpu


## Synthetic Problem Setup
We construct a basic classification objective to evaluate both pipelines side-by-side.

In [2]:
torch.manual_seed(42)
np.random.seed(42)

# Hyperparameters
input_dim = 16
hidden_dim = 32
output_dim = 3
batch_size = 64
num_samples = 1000

# Generate simple synthetic dataset
X_raw = torch.randn(num_samples, input_dim)
true_W = torch.randn(input_dim, output_dim)
Y_raw = torch.argmax(X_raw @ true_W + 0.1 * torch.randn(num_samples, output_dim), dim=1)

# Split into Train and Evaluation sets
train_X, eval_X = X_raw[:800].to(device), X_raw[800:].to(device)
train_Y, eval_Y = Y_raw[:800].to(device), Y_raw[800:].to(device)

## Pipeline 1: Standard Backpropagation
A standard network mapping explicit weight paths optimized using analytical backward automatic differentiation.

In [3]:
class BackpropNet(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hid_dim)
        self.fc2 = nn.Linear(hid_dim, out_dim)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc2(h)

backprop_model = BackpropNet(input_dim, hidden_dim, output_dim).to(device)
optimizer = torch.optim.SGD(backprop_model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

print("Training Pipeline 1 (Standard Backprop)...")
for epoch in range(100):
    backprop_model.train()
    optimizer.zero_grad()

    outputs = backprop_model(train_X)
    loss = criterion(outputs, train_Y)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/100], Loss: {loss.item():.4f}")

Training Pipeline 1 (Standard Backprop)...
Epoch [20/100], Loss: 1.0111
Epoch [40/100], Loss: 0.9311
Epoch [60/100], Loss: 0.8469
Epoch [80/100], Loss: 0.7560
Epoch [100/100], Loss: 0.6633


## Pipeline 2: Dust Optimization Framework (Forward-Only)
This pipeline updates layer matrices by introducing a virtual candidate population via activation-space perturbation vectors, completely bypassing backward tracking graphs.

In [4]:
class DustNet(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hid_dim)
        self.fc2 = nn.Linear(hid_dim, out_dim)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc2(h)

    def forward_perturbed(self, x, epsilon):
        # Perturbing intermediate hidden activation space directly
        h_base = F.relu(self.fc1(x))
        h_perturbed = h_base + epsilon
        return self.fc2(h_perturbed)

dust_model = DustNet(input_dim, hidden_dim, output_dim).to(device)
dust_lr = 0.01
K_populations = 200 # Number of virtual parallel branches
sigma = 0.1         # Perturbation scaling factor

print("Training Pipeline 2 (Dust Alternative - Forward-Only Alpha)...")
for epoch in range(100):
    dust_model.eval() # Disable standard gradient tracking context
    with torch.no_grad():
        # 1. Evaluate baseline trajectory loss
        base_out = dust_model(train_X)
        base_loss = criterion(base_out, train_Y)

        # Determine active intermediate dimension properties
        h_sample = F.relu(dust_model.fc1(train_X))

        # 2. Construct activation tracking noise mappings
        # Shape: [K, Batch, Hidden]
        epsilon = torch.randn(K_populations, train_X.size(0), hidden_dim, device=device) * sigma
        losses = torch.zeros(K_populations, device=device)

        # 3. Assess the parallel target population scores
        for k in range(K_populations):
            out_k = dust_model.forward_perturbed(train_X, epsilon[k])
            losses[k] = criterion(out_k, train_Y)

        # 4. Standardize the loss metrics to isolate direction vectors
        normalized_losses = (losses - base_loss) / (losses.std() + 1e-8)

        # 5. Synthesize projected direction estimate for layer updates
        # Reconstruct an alignment trajectory mapping back to parameter paths
        # Gradient projection alignment approximation step
        grad_est_fc2_w = torch.zeros_like(dust_model.fc2.weight)
        for k in range(K_populations):
            # Coordinate alignment across weight pathways
            grad_est_fc2_w += normalized_losses[k] * (epsilon[k].t() @ base_out).t()[:output_dim, :hidden_dim]

        grad_est_fc2_w /= K_populations

        # Manual parameter update stepping ignoring autograd graphs
        dust_model.fc2.weight.data -= dust_lr * grad_est_fc2_w

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/100], Baseline Loss: {base_loss.item():.4f}")

Training Pipeline 2 (Dust Alternative - Forward-Only Alpha)...
Epoch [20/100], Baseline Loss: 1.0566
Epoch [40/100], Baseline Loss: 1.0554
Epoch [60/100], Baseline Loss: 1.0554
Epoch [80/100], Baseline Loss: 1.0546
Epoch [100/100], Baseline Loss: 1.0527


## Evaluation Pipeline Phase
Evaluating accuracy metrics for both networks side by side against unseen data pools.

In [5]:
def evaluate(model, name):
    model.eval()
    with torch.no_grad():
        preds = torch.argmax(model(eval_X), dim=1)
        accuracy = (preds == eval_Y).float().mean().item() * 100
        print(f"➔ Network Alignment Status [{name}]: Accuracy = {accuracy:.2f}%")
    return accuracy

print("Evaluating training pipelines...")
bp_acc = evaluate(backprop_model, "Pipeline 1: Standard Backprop")
dust_acc = evaluate(dust_model, "Pipeline 2: Dust Alternative")

Evaluating training pipelines...
➔ Network Alignment Status [Pipeline 1: Standard Backprop]: Accuracy = 86.00%
➔ Network Alignment Status [Pipeline 2: Dust Alternative]: Accuracy = 43.00%
